In [1]:
import sqlite3

print("SQLite imported successfully")

SQLite imported successfully


In [2]:
import pandas as pd
import sqlite3

master = pd.read_csv("../data/processed/airbnb_master.csv")

print(master.shape)

(96871, 89)


In [3]:
conn = sqlite3.connect("../database/airbnb.db")

print("SQLite database connected successfully.")

SQLite database connected successfully.


In [4]:
fact_listing_metrics = master[
    [
        "id",
        "host_id",
        "neighbourhood_cleansed",
        "price",
        "availability_365",
        "occupancy_rate",
        "estimated_revenue_l365d",
        "number_of_reviews",
        "review_scores_rating"
    ]
].copy()

dim_listing = master[
    [
        "id",
        "name",
        "room_type",
        "property_type",
        "accommodates",
        "bedrooms",
        "bathrooms",
        "beds"
    ]
].copy()

dim_host = master[
    [
        "host_id",
        "host_name",
        "host_is_superhost",
        "host_listings_count",
        "host_total_listings_count"
    ]
].drop_duplicates()

dim_neighbourhood = master[
    ["neighbourhood_cleansed"]
].drop_duplicates()

print(fact_listing_metrics.shape)
print(dim_listing.shape)
print(dim_host.shape)
print(dim_neighbourhood.shape)

(96871, 9)
(96871, 8)
(55646, 5)
(33, 1)


In [5]:
fact_listing_metrics.to_sql(
    "fact_listing_metrics",
    conn,
    if_exists="replace",
    index=False
)

dim_listing.to_sql(
    "dim_listing",
    conn,
    if_exists="replace",
    index=False
)

dim_host.to_sql(
    "dim_host",
    conn,
    if_exists="replace",
    index=False
)

dim_neighbourhood.to_sql(
    "dim_neighbourhood",
    conn,
    if_exists="replace",
    index=False
)

print("Star schema tables saved to SQLite database.")

Star schema tables saved to SQLite database.


In [6]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,fact_listing_metrics
1,dim_listing
2,dim_host
3,dim_neighbourhood


In [7]:
query = """
SELECT 
    n.neighbourhood_cleansed,
    COUNT(*) AS total_listings,
    ROUND(AVG(f.price), 2) AS avg_price,
    ROUND(AVG(f.occupancy_rate), 2) AS avg_occupancy,
    ROUND(AVG(f.estimated_revenue_l365d), 2) AS avg_revenue
FROM fact_listing_metrics f
JOIN dim_neighbourhood n
    ON f.neighbourhood_cleansed = n.neighbourhood_cleansed
GROUP BY n.neighbourhood_cleansed
ORDER BY avg_revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,neighbourhood_cleansed,total_listings,avg_price,avg_occupancy,avg_revenue
0,Lambeth,5190,345.71,67.30,21777.19
1,Westminster,11385,342.14,55.01,16808.50
2,Kensington and Chelsea,6401,336.07,56.48,15366.72
3,City of London,549,354.39,43.28,13741.30
4,Camden,6551,216.51,59.21,13074.28
5,Islington,5036,217.55,69.54,11709.92
6,Southwark,5475,174.28,65.38,10618.88
7,Hammersmith and Fulham,4157,199.19,62.87,10531.26
8,Tower Hamlets,7469,430.91,66.55,9679.06
9,Richmond upon Thames,1290,184.27,63.63,8509.24


In [8]:
query = """
SELECT 
    l.room_type,
    COUNT(*) AS total_listings,
    ROUND(AVG(f.price), 2) AS avg_price,
    ROUND(AVG(f.occupancy_rate), 2) AS avg_occupancy
FROM fact_listing_metrics f
JOIN dim_listing l
    ON f.id = l.id
GROUP BY l.room_type
ORDER BY avg_price DESC;
"""

pd.read_sql(query, conn)

,room_type,total_listings,avg_price,avg_occupancy
0,Hotel room,109,657.83,47.53
1,Entire home/apt,62907,279.35,59.17
2,Private room,33643,121.71,62.55
3,Shared room,212,96.91,37.66
